# Sleep states with HED and ontology links: DANDI:000939

This notebook runs the `dandi_000939` conversion on one published session of the Peyrache lab
head-direction dataset ([DANDI:000939](https://dandiarchive.org/dandiset/000939/0.261016.2310),
Duszkiewicz et al., 2024) and shows what it adds.

The published files store sleep scoring as a `TimeIntervals` table, `/intervals/sleep_states`, with one
row per continuous period of wake, NREM or REM sleep. The `state` column holds the labels `wake`, `nrem`
and `rem`, but nothing in the file says what those labels mean.

The conversion reads the file, adds three things, and exports the result to a new file with the same name.
The `sleep_states` columns and all other data are copied unchanged:

1. `HedLabMetaData`, which records the HED schema versions the annotations use.
2. A `MeaningsTable` bound to the `state` column, named `state_meanings` (hdmf-common names a
   MeaningsTable after the column it describes). It has one row per label, a written `meaning`, and a
   `HED` column with a machine-readable [HED](https://www.hedtags.org/) annotation. The tags follow the
   proposed NWB best practice for sleep states
   ([nwbinspector#775](https://github.com/NeurodataWithoutBorders/nwbinspector/pull/775), open at the
   time of writing).
3. HERD references (`/general/external_resources`) that link each label to a term in the
   [Neuro Behavior Ontology](https://bioregistry.io/registry/nbo) (NBO). This is the same mechanism
   NeuroConv uses for species, strain and brain areas
   ([NeuroConv ontology guide](https://neuroconv.readthedocs.io/en/main/user_guide/ontology.html)).

## Setup

From the root of the `peyrache-lab-to-nwb` repository:

```bash
uv pip install -e .
```

This installs `ndx-hed` 0.3.0 (which requires `pynwb>=4.2.0` and `hdmf>=6.2.0`) and the DANDI client.

In [1]:
import subprocess
import sys
from pathlib import Path

import h5py
import pandas as pd
from IPython.display import HTML, display
from pynwb import NWBHDF5IO

from ndx_hed.utils.hed_nwb_validator import HedNWBValidator
from peyrache_lab_to_nwb.dandi_000939.convert_session import load_metadata, session_to_nwb

## 1. The annotations, as defined in `metadata.yaml`

The conversion reads the HED schema versions, the meaning, HED annotation and ontology term of each state
from `metadata.yaml` next to `convert_session.py`.

- `wake` → `Awake`
- `nrem` → `Asleep`
- `rem` → `Asleep, sc:Sleep-stage-REM`

The standard HED schema has `Awake` and `Asleep` but no NREM or REM tag. REM comes from the SCORE library
(prefix `sc:`). NREM is tagged `Asleep` alone, because the lab scored NREM as one state and did not divide
it into the human stages N1 to N3.

In [2]:
metadata = load_metadata()
print("HED schema version:", metadata["HedLabMetaData"]["hed_schema_version"])
pd.DataFrame(
    [
        dict(value=value, meaning=state["meaning"], HED=state["HED"], ontology_term=state["ontology_term"]["id"])
        for value, state in metadata["SleepStateMeanings"]["states"].items()
    ]
)

HED schema version: ["8.4.0", "sc:score_2.1.0"]


,value,meaning,HED,ontology_term
0,wake,Wakefulness.,Awake,NBO:0000066
1,nrem,Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161
2,rem,Rapid eye movement (REM) sleep.,"Asleep, sc:Sleep-stage-REM",NBO:0000159


## 2. Download a session and add the annotations

`session_to_nwb` does the whole conversion for one session:

1. Downloads the published file to `{download_dir_path}/{asset_path}`, or reuses it if it is already there.
   We use subject A1821 (23.6 GB). The downloaded file is only read.
2. Reads it and adds `HedLabMetaData`, the `state_meanings` MeaningsTable and the HERD references. The HED
   annotations are validated before the file is written.
3. Exports everything to a new file, `{output_dir_path}/{asset_path}`. The export copies all data, with the
   same chunking and compression, and caches only the current schemas (explained in section 4). For this
   session it takes about 30 s and writes 23.6 GB.

Set `download_dir_path` and `output_dir_path` to folders on your machine.

In [3]:
asset_path = "sub-A1821/sub-A1821_ses-190910_behavior+ecephys+ogen.nwb"
download_dir_path = Path.home() / "source_data" / "dandi" / "000939"
output_dir_path = Path.home() / "nwb_output" / "dandi_000939"

annotated_path = session_to_nwb(
    asset_path=asset_path,
    output_dir_path=output_dir_path,
    download_dir_path=download_dir_path,
    overwrite=True,
)
published_path = download_dir_path / asset_path

Wrote /Users/weian/nwb_output/dandi_000939/sub-A1821/sub-A1821_ses-190910_behavior+ecephys+ogen.nwb


## 3. The published file

Each row of `sleep_states` is one continuous period in one state. The `tags` column is a zero-based running
count of that row's state (wake 0, nrem 0, wake 1, ...). We checked this on all 31 files in the Dandiset.

In [4]:
with NWBHDF5IO(published_path, mode="r") as io:
    published_sleep_states = io.read().intervals["sleep_states"].to_dataframe()

print("Columns:", list(published_sleep_states.columns))
published_sleep_states.head(15)

Columns: ['start_time', 'stop_time', 'state', 'tags']


,start_time,stop_time,state,tags
id,,,,
0,1.0,57.0,wake,[0]
1,58.0,68.0,nrem,[0]
2,69.0,144.0,wake,[1]
3,145.0,157.0,nrem,[1]
4,158.0,173.0,wake,[2]
5,174.0,283.0,nrem,[2]
6,284.0,294.0,wake,[3]
7,295.0,691.0,nrem,[3]
8,692.0,705.0,wake,[4]


In [5]:
published_sleep_states["duration"] = published_sleep_states["stop_time"] - published_sleep_states["start_time"]
published_sleep_states.groupby("state")["duration"].agg(n_periods="count", total_s="sum").round(1)

,n_periods,total_s
state,,
nrem,39,4944.0
rem,10,324.0
wake,48,6185.0


## 4. Cached schemas

An NWB file caches the schemas it was written with under `/specifications`. The published file was written
with NWB 2.9.0 and caches hdmf-common 1.8.0. `MeaningsTable` is an hdmf-common 1.10.0 type.

In [6]:
def cached_namespaces(path):
    with h5py.File(path, "r") as file:
        return {name: sorted(file["specifications"][name].keys()) for name in file["specifications"]}


for label, path in [("Published", published_path), ("Annotated", annotated_path)]:
    with h5py.File(path, "r") as file:
        print(f"{label}: NWB {file.attrs['nwb_version']}, cached {cached_namespaces(path)}")

Published: NWB 2.9.0, cached {'core': ['2.9.0'], 'hdmf-common': ['1.8.0'], 'hdmf-experimental': ['0.5.0']}
Annotated: NWB 2.11.0, cached {'core': ['2.11.0'], 'hdmf-common': ['1.10.0'], 'hdmf-experimental': ['0.6.0'], 'ndx-hed': ['0.3.0']}


Appending to the published file in place would cache the namespaces of the installed PyNWB (core 2.11.0,
hdmf-common 1.10.0, hdmf-experimental 0.6.0) and ndx-hed 0.3.0 next to the ones already in the file.

With two versions of `core` cached, tools that read only the cached schemas fail to load the file. The
ndx-hed namespace depends on `core` without naming a version, and the loader resolves it against core 2.9.0,
which lacks the `DurationVectorData` type that ndx-hed uses:

```
ValueError: No specification for 'DurationVectorData' in namespace 'ndx-hed'
```

PyNWB's validator, NWB Inspector and `dandi validate` all read the cached schemas and fail this way. The
export writes a new file, which caches only one version of each namespace and sets `nwb_version` to 2.11.0,
as shown above.

We validate the annotated file against the schemas cached in it, which is how DANDI and NWB Inspector read it.

In [7]:
result = subprocess.run(
    [sys.executable, "-m", "pynwb.validation_cli", str(annotated_path)], capture_output=True, text=True
)
print(result.stdout or result.stderr)

Validating /Users/weian/nwb_output/dandi_000939/sub-A1821/sub-A1821_ses-190910_behavior+ecephys+ogen.nwb against cached namespace information using namespace 'ndx-hed'.
 - no errors found.



## 5. The annotated file

In [8]:
io = NWBHDF5IO(annotated_path, mode="r")
annotated_nwbfile = io.read()
annotated_sleep_states = annotated_nwbfile.intervals["sleep_states"]

### The intervals table

The columns and rows are the same as in the published file. Code that reads `sleep_states` today keeps
working.

In [9]:
print("Columns:", annotated_sleep_states.colnames)
intervals_df = annotated_sleep_states.to_dataframe()
print("Rows unchanged:", intervals_df.equals(published_sleep_states.drop(columns="duration")))
intervals_df.head(15)

Columns: ('start_time', 'stop_time', 'state', 'tags')
Rows unchanged: True


,start_time,stop_time,state,tags
id,,,,
0,1.0,57.0,wake,[0]
1,58.0,68.0,nrem,[0]
2,69.0,144.0,wake,[1]
3,145.0,157.0,nrem,[1]
4,158.0,173.0,wake,[2]
5,174.0,283.0,nrem,[2]
6,284.0,294.0,wake,[3]
7,295.0,691.0,nrem,[3]
8,692.0,705.0,wake,[4]


### The MeaningsTable

`get_meanings_for_column("state")` returns the MeaningsTable bound to the `state` column. It is named
`state_meanings` and stored in the `meanings_tables` group of `sleep_states`. Its `target` is an object
reference to the `state` column.

In [10]:
meanings = annotated_sleep_states.get_meanings_for_column("state")
print("Name:  ", meanings.name)
print("Target:", meanings.target.name, "column of", meanings.target.parent.name)
print("Description:", meanings.description)
meanings.to_dataframe()

Name:   state_meanings
Target: state column of sleep_states
Description: Meaning of each sleep state label. Scored with the automated SleepScoreMaster algorithm and manually checked in TheStateEditor (Buzsaki lab), using LFP and a pseudo-EMG derived from correlated high-frequency activity across probe channels (Duszkiewicz et al., 2024; Watson et al., 2016).


,value,meaning,HED
id,,,
0,wake,Wakefulness.,Awake
1,nrem,Non-rapid eye movement (NREM) sleep.,Asleep
2,rem,Rapid eye movement (REM) sleep.,"Asleep, sc:Sleep-stage-REM"


### The HED schema versions

`HedLabMetaData` sits in `/general` and records which HED schemas the annotations use.

In [11]:
hed_lab_metadata = annotated_nwbfile.get_lab_meta_data("hed_schema")
print("HED schema version:", hed_lab_metadata.hed_schema_version)

HED schema version: ["8.4.0", "sc:score_2.1.0"]


### The ontology links

`nwbfile.external_resources` is the HERD stored under `/general/external_resources`. Each row links one
key, a label in the MeaningsTable `value` column, to one ontology term.

In [12]:
resources_df = annotated_nwbfile.external_resources.to_dataframe()
# Resolve each object_id to the table and column it points to
resources_df["linked_column"] = [
    f"{annotated_nwbfile.objects[object_id].parent.name}/{annotated_nwbfile.objects[object_id].name}"
    for object_id in resources_df["object_id"]
]
resources_df = resources_df[["object_type", "linked_column", "key", "entity_id", "entity_uri"]].reset_index(drop=True)

display_df = resources_df.copy()
display_df["entity_uri"] = display_df["entity_uri"].apply(lambda uri: f'<a href="{uri}" target="_blank">{uri}</a>')
display(HTML(display_df.to_html(escape=False, index=False)))

object_type,linked_column,key,entity_id,entity_uri
VectorData,state_meanings/value,wake,NBO:0000066,http://purl.obolibrary.org/obo/NBO_0000066
VectorData,state_meanings/value,nrem,NBO:0000161,http://purl.obolibrary.org/obo/NBO_0000161
VectorData,state_meanings/value,rem,NBO:0000159,http://purl.obolibrary.org/obo/NBO_0000159


In [13]:
ONTOLOGY_INFO = {"NBO": ("Neuro Behavior Ontology", "Behavior processes and phenotypes, including sleep and wakefulness.")}

prefixes = sorted({entity_id.split(":")[0] for entity_id in resources_df["entity_id"]})
rows = []
for prefix in prefixes:
    name, description = ONTOLOGY_INFO.get(prefix, (prefix, ""))
    url = f"https://bioregistry.io/registry/{prefix.lower()}"
    rows.append(f'<li><a href="{url}" target="_blank">{name}</a> (<code>{prefix}</code>): {description}</li>')
display(HTML("<ul>" + "".join(rows) + "</ul>"))

### The rest of the file

The append leaves the other data in place, for example the raw signal and the units.

In [14]:
print("Raw signal:", annotated_nwbfile.acquisition["ElectricalSeries"].data.shape)
print("Units:     ", len(annotated_nwbfile.units))
print("Epochs:    ", list(annotated_nwbfile.epochs.to_dataframe()["tags"].str[0]))

Raw signal: (231036480, 64)
Units:      74
Epochs:     ['home_cage', 'wake_square', 'home_cage', 'wake_opto']


## 5. Using the annotations

### Validate the HED annotations

`HedNWBValidator` checks every HED string in the file, including the HED column of the MeaningsTable,
against the schemas named in `HedLabMetaData`.

In [15]:
issues = HedNWBValidator(hed_lab_metadata).validate_file(annotated_nwbfile)
print("HED validation issues:", len(issues))

HED validation issues: 0


### One view of intervals, meanings, HED and ontology terms

Joining the MeaningsTable and the HERD terms onto the intervals gives each period its definition, its HED
annotation and its ontology term.

In [16]:
terms = resources_df[["key", "entity_id"]].rename(columns={"key": "value"})
lookup = meanings.to_dataframe().merge(terms, on="value")
annotated_df = intervals_df.merge(lookup, left_on="state", right_on="value", how="left").drop(columns="value")
annotated_df.head(15)

,start_time,stop_time,state,tags,meaning,HED,entity_id
0,1.0,57.0,wake,[0],Wakefulness.,Awake,NBO:0000066
1,58.0,68.0,nrem,[0],Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161
2,69.0,144.0,wake,[1],Wakefulness.,Awake,NBO:0000066
3,145.0,157.0,nrem,[1],Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161
4,158.0,173.0,wake,[2],Wakefulness.,Awake,NBO:0000066
5,174.0,283.0,nrem,[2],Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161
6,284.0,294.0,wake,[3],Wakefulness.,Awake,NBO:0000066
7,295.0,691.0,nrem,[3],Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161
8,692.0,705.0,wake,[4],Wakefulness.,Awake,NBO:0000066
9,706.0,713.0,nrem,[4],Non-rapid eye movement (NREM) sleep.,Asleep,NBO:0000161


### Select intervals by HED tag

A tool that knows nothing about this lab's labels can still find all sleep periods, because both `nrem` and
`rem` carry the HED tag `Asleep`.

In [17]:
is_asleep = annotated_df["HED"].str.split(", ").apply(lambda tags: "Asleep" in tags)
asleep_df = annotated_df[is_asleep]
asleep_s = (asleep_df["stop_time"] - asleep_df["start_time"]).sum()
print(f"{len(asleep_df)} sleep periods, {asleep_s:.0f} s in total")
asleep_df.groupby("state").size()

49 sleep periods, 5268 s in total


state
nrem    39
rem     10
dtype: int64

In [18]:
io.close()

## Summary

- `session_to_nwb` downloads a published session, adds HED metadata, a MeaningsTable and HERD references,
  and exports it to a new file that caches only the current schemas.
- The published `sleep_states` table keeps its four columns and its rows.
- The `state_meanings` MeaningsTable defines each label in words and with HED (`Awake`, `Asleep`,
  `Asleep, sc:Sleep-stage-REM`), validated against HED 8.4.0 and SCORE 2.1.0.
- HERD references link each label to a Neuro Behavior Ontology term.

### Further resources

- [Proposed best practice for sleep states, nwbinspector#775](https://github.com/NeurodataWithoutBorders/nwbinspector/pull/775)
- [ndx-hed](https://github.com/hed-standard/ndx-hed), see `examples/03_events_table_integration.py`
- [HED schema browser](https://www.hedtags.org/hed-schema-browser/)
- [NeuroConv ontology annotation guide](https://neuroconv.readthedocs.io/en/main/user_guide/ontology.html)
- [DANDI:000939](https://dandiarchive.org/dandiset/000939/0.261016.2310)